## Proton Mail Trash Review
Run the notebook from top to bottom. This notebook contains only the normal workflow, not the pagination diagnostics or full-Trash scan experiments.
The workflow should inspect at most two recent Trash pages and at most five selected conversations. If a cell fails, stop at the first traceback; do not immediately rerun the model.

In [9]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SRC_ROOT = PROJECT_ROOT / "src"

if not (SRC_ROOT / "agentic_ai").is_dir():
    raise RuntimeError(
        f"Expected the agentic-ai project directory, got: {PROJECT_ROOT}"
    )

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_ROOT)


Project root: /Users/theo/Documents/Projects/programingProjects/agentic_AI_project/agentic-ai
Source directory: /Users/theo/Documents/Projects/programingProjects/agentic_AI_project/agentic-ai/src


In [2]:
import importlib
import inspect

import agentic_ai.models as models_module
import agentic_ai.proton as proton_module
import agentic_ai.candidate_selection as candidate_selection_module
import agentic_ai.review_validation as review_validation_module
import agentic_ai.agent as agent_module
import agentic_ai.workflow as workflow_module

# Reload in dependency order: dependencies first, workflow last.
importlib.reload(models_module)
importlib.reload(proton_module)
importlib.reload(candidate_selection_module)
importlib.reload(review_validation_module)
importlib.reload(agent_module)
importlib.reload(workflow_module)

# Refresh notebook-level references after reloading.
from agentic_ai.workflow import review_trash
from agentic_ai.proton import (
    close_proton_mail,
    ensure_logged_in,
    open_proton_mail,
    open_trash,
)

print("Draft model loaded:", hasattr(models_module, "TrashReviewReportDraft"))
print("Notebook function is current:", review_trash is workflow_module.review_trash)
print(
    "Candidate factory is current:",
    workflow_module.create_candidate_selection_agent
    is agent_module.create_candidate_selection_agent,
)
print(
    "Review factory is current:",
    workflow_module.create_trash_review_agent
    is agent_module.create_trash_review_agent,
)
print(
    "Bounded request settings found:",
    inspect.getsource(workflow_module.review_trash).count("request_limit=2"),
)


Draft model loaded: True
Notebook function is current: True
Candidate factory is current: True
Review factory is current: True
Bounded request settings found: 2


In [10]:
def print_trash_review(report) -> None:
    print("Trash review")
    print("=" * 50)

    if not report.findings:
        print("No conversations need checking.")
        return

    print(f"{len(report.findings)} conversation(s) need checking.")
    print()

    for number, finding in enumerate(report.findings, start=1):
        print(f"{number}. Conversation {finding.conversation_index}")
        print(f"   Category: {finding.category}")
        if finding.event_date is not None:
            print(f"   Event date: {finding.event_date}")
        print(f"   Reason: {finding.reason}")
        print(f"   Evidence: {finding.evidence}")
        print()


## Open Proton Mail
Run this cell once. Complete login manually in the browser if prompted. It opens Trash and does not scan or modify email by itself.

In [4]:
playwright, context, page = await open_proton_mail(PROJECT_ROOT)
print("Browser objects created:", page is not None)

await ensure_logged_in(page)
await open_trash(page)
print("Ready to review Trash:", page.url)


Browser objects created: True
Proton Mail login is required.
Log in manually in the browser window.
Login detected.
Opened Trash.
Ready to review Trash: https://mail.proton.me/u/1/trash


## Run the bounded review
Execute this cell once. The workflow should collect the two most recent Trash pages, select up to five candidate conversations, inspect them, and validate the report.

In [5]:
run = await review_trash(page)

print(f"Analysis date: {run.analysis_date}")
print(f"Candidates: {run.candidate_indexes}")
print(f"Inspected: {run.inspected_indexes}")
print()
print_trash_review(run.report)


                 pydantic-ai v2.50.0 • Python 3.13.15

      / \        agent: candidate_agent • model: ollama:qwen3:4b-instruct
     /   \         output: TrashCandidateSelection • tools: 0 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


Returned to Trash.
Returned to Trash.
Returned to Trash.
Returned to Trash.
Returned to Trash.


ValueError: Conversation 11: event date 2026-10-14 is not in the future.

## Close the browser
Run after inspecting the report. If the workflow failed, you can also run this cleanup cell once you no longer need the browser.

In [6]:
import re

from agentic_ai.proton import (
    is_trash_page,
    open_trash,
    open_trash_conversation,
    return_to_trash,
)

if not await is_trash_page(page):
    await open_trash(page)

content = await open_trash_conversation(page, index=36)

try:
    print(f"Conversation index: {content.conversation_index}")
    print()

    lines = content.raw_text.splitlines()

    for i, line in enumerate(lines):
        if re.search(
            r"2026|October|September|August|July|June|May|April",
            line,
            re.IGNORECASE,
        ):
            start = max(0, i - 1)
            end = min(len(lines), i + 2)

            print(f"--- Lines {start + 1}–{end} ---")
            print("\n".join(lines[start:end]))
            print()
finally:
    await return_to_trash(page)

Conversation index: 36

--- Lines 73–75 ---
Monday
Monday, October 12th, 2026 at 12:29 PM
To

Returned to Trash.


In [ ]:
if "playwright" in globals() and "context" in globals():
    await close_proton_mail(playwright, context)
    print("Proton browser session closed.")
else:
    print("No browser session is available to close.")


In [ ]:
import importlib
import agentic_ai.workflow as workflow_module

importlib.reload(workflow_module)

print(
    "Function exists in workflow.py:",
    hasattr(workflow_module, "remove_email_header_timestamps"),
)

from agentic_ai.workflow import remove_email_header_timestamps

sample = """Monday
Monday, October 12th, 2026 at 12:29 PM
To
recipient@example.com

The meeting will take place on October 20th, 2026."""

cleaned = remove_email_header_timestamps(sample)

print(cleaned)

assert "Monday, October 12th, 2026 at 12:29 PM" not in cleaned
assert "The meeting will take place on October 20th, 2026." in cleaned

print("Timestamp-filter test passed.")

Function exists in workflow.py: True
To
recipient@example.com

The meeting will take place on October 20th, 2026.
Timestamp-filter test passed.


In [14]:
import importlib
import agentic_ai.workflow as workflow_module

importlib.reload(workflow_module)

from agentic_ai.workflow import review_trash

print("Workflow reloaded:", review_trash is workflow_module.review_trash)

Workflow reloaded: True


In [15]:
run = await review_trash(page)

print(f"Analysis date: {run.analysis_date}")
print(f"Candidates: {run.candidate_indexes}")
print(f"Inspected: {run.inspected_indexes}")

print()
print_trash_review(run.report)

Returned to Trash.
Returned to Trash.
Returned to Trash.
Returned to Trash.
Returned to Trash.
Analysis date: 2026-10-14
Candidates: [11, 36, 93, 94, 95]
Inspected: [11, 36, 93, 94, 95]

Trash review
2 conversation(s) need checking.

1. Conversation 11
   Category: unusual
   Reason: This message is a job application from Region Midtjylland. It is the only message in the conversation with a clear job-related subject and is not clearly deviating from the surrounding messages. However, it is a standalone application message in a context of job applications, and its content is distinct from other messages which are either invitations or confirmations. This deviation from the pattern of other messages (which are either invitations or confirmations) makes it a candidate for human review.
   Evidence: Din ansøgning til stillingen som IT-specialist til udvikling og automatisering i Netværk i Digitalisering og It i Region Midtjylland

2. Conversation 36
   Category: unusual
   Reason: This mes